# Domando al Tren de Carga (Intro a PySpark)

**Contexto (El Corazon del Problema):** Anteriormente trabajamos con `mrjob`. Entendiste como tomar los datos (Map) y volver a unirlos (Reduce) porque no caben en la RAM de un solo computador.

Pero la industria real ya no usa discos duros para esto (es muy lento). Hoy conoceremos a **Apache Spark**: el motor de procesamiento distribuido in-memory que domina el mundo (Uber, Netflix, NASA, ...).

**La Mision de Hoy:** 
Eres un Cientifico de Datos recien contratado en un gigante del E-Commerce ('ShopDist'). Te han pedido analizar 5 millones de logs de clicks de usuarios. Instalarás Spark en tu 'espacio de trabajo' y descubrirás una verdad técnica dolorosa: **A veces, la herramienta de Big Data es mas lenta que la herramienta normal.**

---
### Pre-Requisitos Técnicos
`uv pip install pyspark`

PySpark necesita Java por debajo para funcionar (la máquina virtual de Scala). Ejecuta la siguiente celda para comprobar que nuestro entorno Linux ya lo tiene.

In [2]:
!java -version

openjdk version "25.0.1" 2025-10-21 LTS
OpenJDK Runtime Environment Microsoft-12574222 (build 25.0.1+8-LTS)
OpenJDK 64-Bit Server VM Microsoft-12574222 (build 25.0.1+8-LTS, mixed mode, sharing)


---
## PARTE 1: Encendiendo el Motor
Para usar Spark en Python, no usamos la librería en directo. Necesitamos crear una **SparkSession**. Es literalmente un puente telefónico entre tu codigo Python y un cluster de Java (que nosotros simularemos usando todos los núcleos de nuestro procesador local `local[*]`).

In [3]:
from pyspark.sql import SparkSession
import time
import pandas as pd
import numpy as np

# INICIA TU CODIGO AQUI
# Construye la sesion. Ponle tu nombre a la aplicacion en appName()
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Analisis_Albeiro') \
    .getOrCreate()
# TERMINA TU CODIGO AQUI

print('¡Motor de Spark Encendido!')
print('Version corriendo:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/10 20:18:13 WARN Utils: Your hostname, codespaces-3f0260, resolves to a loopback address: 127.0.0.1; using 10.0.1.223 instead (on interface eth0)
26/04/10 20:18:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:18:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


¡Motor de Spark Encendido!
Version corriendo: 4.1.1


---
## PARTE 2: La Paradoja de la Velocidad (Pandas vs Spark)

**La Regla de Oro:** "No contrates un tren de carga de 100 vagones para ir a la esquina a comprar pan".

Spark tiene un 'Overhead' enorme: Para hacer cualquier suma, debe traducir tu codigo Python a Java, crear tareas de red, revisar los nodos (núcleos) y devolver el resultado. Mientras, Pandas (`C` puro continuo) no tiene esa burocracia.

Vamos a generar 3 millones de registros financieros en la RAM para probarlo.

In [4]:
# Generador Sintetico de ShopDist
np.random.seed(42)
n_filas = 3000000

# Creamos un DataFrame de Pandas clasico
df_pandas = pd.DataFrame({
    'id_usuario': np.random.randint(1, 1000, n_filas),
    'gasto_usd': np.random.uniform(1.0, 500.0, n_filas)
})
print(f'Creado dataset en Pandas con {n_filas} filas.')

# Ahora le pedimos a Spark que 'ingiera' ese Pandas y lo convierta en un DataFrame Distribuido
print('Convirtiendo a Spark (Esto puede tardar un poco por la serializacion...)')
df_spark = spark.createDataFrame(df_pandas)
print('Listo.')

Creado dataset en Pandas con 3000000 filas.
Convirtiendo a Spark (Esto puede tardar un poco por la serializacion...)


26/04/10 20:18:24 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

Listo.


### Cronómetro
Calculemos el promedio global de la columna `gasto_usd`.

In [5]:
# TEST 1: PANDAS (Un solo hilo, pura memoria C)
inicio = time.time()
promedio_p = df_pandas['gasto_usd'].mean()
fin = time.time()
tiempo_pandas = fin - inicio
print(f'Pandas promedio: {promedio_p:.2f} | Tiempo: {tiempo_pandas:.5f} seg')

# TEST 2: SPARK (Multiples hilos simulados, Overhead de Java)
import pyspark.sql.functions as F

inicio = time.time()
promedio_s = df_spark.select(F.mean('gasto_usd')).collect()[0][0]
fin = time.time()
tiempo_spark = fin - inicio
print(f'Spark promedio : {promedio_s:.2f} | Tiempo: {tiempo_spark:.5f} seg')

print(f'\n>> Conclusión matematica: Spark fue {tiempo_spark/tiempo_pandas:.1f} veces mas lento en esta maquina pequeña.')

Pandas promedio: 250.48 | Tiempo: 0.00549 seg


26/04/10 20:19:18 WARN TaskSetManager: Stage 0 contains a task of very large size (20185 KiB). The maximum recommended task size is 1000 KiB.


Spark promedio : 250.48 | Tiempo: 6.58644 seg

>> Conclusión matematica: Spark fue 1200.8 veces mas lento en esta maquina pequeña.


**PREGUNTA (Analisis Estadistico Moderno):**
Si Spark es mucho mas lento con 3 millones de datos... ¿En que punto crítico de la arquitectura crees que Spark destrozaría a Pandas y se volveria imprescindible? (Discútelo en clase).

---
## PARTE 3: La Evaluacion Perezosa (El Truco Mágico de Spark)
En Pandas, si haces un filtro, Pandas recorre la tabla de inmediato. 
En Spark, existen **Transformaciones** (Anotar la receta en un papel) y **Acciones** (Meter al horno).

Comprueba esto midiendo el tiempo de las celdas.

In [7]:
# TRANSFORMACION PEREZOSA
inicio = time.time()
# Le pedimos a Spark una operacion pesada: Filtra gastos mayores a 200 y suma un impuesto
df_filtrado = df_spark.filter(F.col('gasto_usd') > 200.0).withColumn('impuesto', F.col('gasto_usd') * 1.19)
fin = time.time()

print(f'Tiempo de crear la receta compleja: {fin - inicio:.5f} seg')
print('¡Casi cero segundos! Spark es un genio engañando. Solo construyo un Grafo (DAG), no toco los datos.')

Tiempo de crear la receta compleja: 0.03241 seg
¡Casi cero segundos! Spark es un genio engañando. Solo construyo un Grafo (DAG), no toco los datos.


In [8]:
# ACCION VERDADERA
inicio = time.time()
# INICIA TU CODIGO AQUI
# Invoca la Accion .count() sobre el df_filtrado que creamos arriba
cantidad_usuarios = df_filtrado.count()
# TERMINA TU CODIGO AQUI

fin = time.time()
print(f'Cantidad filtrada: {cantidad_usuarios} usuarios.')
print(f'Tiempo de EJECUTAR la accion real (hornear el plato): {fin - inicio:.5f} seg')

26/04/10 20:19:33 WARN TaskSetManager: Stage 3 contains a task of very large size (20185 KiB). The maximum recommended task size is 1000 KiB.


Cantidad filtrada: 1803498 usuarios.
Tiempo de EJECUTAR la accion real (hornear el plato): 3.86840 seg


---
## PARTE 4: La Sintaxis Declarativa (Dile adiós a mrjob)
Spark sigue haciendo el famoso `Map`, `Shuffle` y `Reduce` por debajo de las sabanas (en la JVM), pero a ti como estadístico, te expone una API hermosa y familiar.

| Tarea a realizar | PANDAS (Local, Memoria C) | PySPARK (Distribuido, Evaluacion Perezosa) |
| :--- | :--- | :--- |
| **Ver Schema** | `df.info()` | `df.printSchema()` |
| **Ver Datos (TOP 5)** | `df.head(5)` | `df.show(5)` |
| **Filtrar filas** | `df[df['A'] > 5]` | `df.filter(F.col('A') > 5)` |
| **Agrupar y sumar** | `df.groupby('B')['A'].sum()` | `df.groupBy('B').agg(F.sum('A'))` |

**RETO FINAL:**
Quieres saber, en el DataFrame `df_spark`, cual fue la Facturacion Total (suma del `gasto_usd`) por cada `id_usuario`. 
Y como esto es Spark, quieres ver en pantalla (show) los 5 usuarios mas gastadores de todos.

In [9]:
# INICIA TU CODIGO AQUI (Basate en la tabla comparativa de arriba)
# Recuerda: Agrupa, suma la columna, ordenalo descendente (.orderBy) y muestra 5 registros.
resultado_final = (
    df_spark
    .groupBy('id_usuario')   # Pon la agrupacion por usuario
    .agg(F.sum('gasto_usd').alias('total_gastado'))   # Aplica la funcion agregada Suma (F.sum) sobre el gasto
    .orderBy(F.col('total_gastado').desc()) # Ya te damos el sort distribuido
)

# Aplica la ACCION de visualizacion (show de 5)
resultado_final.show(5)
# TERMINA TU CODIGO AQUI


26/04/10 20:19:40 WARN TaskSetManager: Stage 6 contains a task of very large size (20185 KiB). The maximum recommended task size is 1000 KiB.


+----------+-----------------+
|id_usuario|    total_gastado|
+----------+-----------------+
|       849|801023.8417435223|
|       265| 799346.765780482|
|       295|799285.8431462727|
|       898|796758.0535000069|
|       727| 793766.725214575|
+----------+-----------------+
only showing top 5 rows


### Para finalizar, cierra siempre tu cluster cuando termines de investigar:
Esto devuelve la RAM al servidor principal.

In [10]:
spark.stop()